In [104]:
import pandas as pd
import numpy as np
from pathlib import Path
from plots import compute_nmdt


F1_METRIC = "dd.my_f1"
logs = Path("../logs")
tables = Path("../tables")

In [105]:
df_dd = pd.read_csv(logs / "dd_run/data_frame.csv")
df_ev = pd.read_csv(logs / "evaluate/data_frame.csv")

dd_best = df_dd.groupby(
    [
        "boundary",
        "strategy",
        "detector",
    ]
)[F1_METRIC].mean()
best_methods = dd_best.groupby(["boundary", "strategy"]).idxmax()
best_detectors = (
    df_ev[["boundary", "strategy", "detector_label"]]
    .apply(tuple, axis=1)
    .isin(best_methods.values)
)
mask = (
    (df_ev["detector_label"] == "oracle") |
    (df_ev["detector_label"] == "ADWIN_JOINT") |
    (df_ev["detector_label"] == "ADWIN")
    # (df_ev["detector_label"] == "STEPD")

)
# df_ev.loc[best_detectors, "detector_label"] = "Best"
df_ev = df_ev[mask]
df_ev["detector_label"] = df_ev["detector_label"].replace("ADWIN_JOINT", "ADWIN*")
df_ev = compute_nmdt(df_ev)

In [106]:
from typing import Optional, Literal
from jinja2 import Template

_template = Template(
    r"{% if font_style %}\{{font_style ~ '{'}}{% endif %}"
    r"{{ '{:.{}f}'.format(mean, ndigits) }}\tiny{$\pm${{ '{:.{}f}'.format(std, ndigits) ~ '}' }}"
    r"{% if font_style %}}{% endif %}"
)

TexFontStyle = Literal["textmd", "textbf", "textup", "textit", "textsl", "textsc"]


def format_column(
    df: pd.DataFrame | pd.Series,
    mean_col="mean",
    std_col="std",
    min_style: Optional[TexFontStyle] = None,
    max_style: Optional[TexFontStyle] = None,
    ndigits=3,
    nan_str="N/A",
) -> pd.DataFrame | pd.Series:
    col_max = df[mean_col].max()
    col_min = df[mean_col].min()

    def _func(row: pd.Series) -> str:
        mean = float(row[mean_col])
        std = float(row[std_col])
        font_style = None

        if pd.isna(mean) or pd.isna(std):
            return nan_str

        # Perform the comparison using the same precision as the formatted output
        if max_style and round(mean, ndigits) == round(col_max, ndigits):
            font_style = max_style
        elif min_style and round(mean, ndigits) == round(col_min, ndigits):
            font_style = min_style

        return _template.render(
            mean=mean,
            std=std,
            font_style=font_style,
            ndigits=ndigits,
        )

    return df.apply(_func, axis=1)

## Significance Test

In the tables, statistically significant deviation from the control (`Oracle') is indicated by an $\uparrow$ or $\downarrow$.
To assess this, we performed a Welch's ANOVA with Benjamini-Hochberg multiple comparison correction, followed by a Games-Howell test ($p=0.05$)~\cite{Vallat2018}.
This analysis was performed for each combination of boundary and strategy type, comparing 'Best' and 'ADWIN*' against the 'Oracle' control.
We opted for Games-Howell over the common Tukey HSD because it is more robust to heterogeneous variances.

* 18 contexts.
    * 3 groups: Control, Method A, Method B
        * 10 samples in each group.



In [107]:
import pingouin as pg
from typing import Set

significance_level = 0.05
dv = "ocl.accuracy_seen_avg"
between = "detector_label"
significantly_better: Set[tuple[str, str, str]] = set()
significantly_worse: Set[tuple[str, str, str]] = set()

# ANOVA STATISTICAL TESTS
anova_results = []

for i, (keys, context) in enumerate(df_ev.groupby(["boundary", "strategy"])):
    if keys[1] == "FT":
        continue

    # Statistical Test: ANOVA
    df_anova = pg.anova(data=context, dv=dv, between=between)
    anova_p = df_anova["p_unc"].values[0]

    anova_results.append(
        {"index": i, "keys": keys, "context": context, "p_unc": anova_p}
    )

# Adjust ANOVA for multiple comparisons
# Extract the raw p-values
p_vals = [res["p_unc"] for res in anova_results]

# Apply Benjamini-Hochberg False Discovery Rate (FDR) correction
# You can change 'fdr_bh' to 'bonf' for stricter Bonferroni correction
reject, p_cor = pg.multicomp(p_vals, alpha=significance_level, method="fdr_bh")

# POST-HOC TESTS
# Run post-hoc tests on significant contexts
for i, res in enumerate(anova_results):
    # Check the boolean 'reject' array from pg.multicomp
    if not reject[i]:
        print(f"{res['index']} {res['keys']} NOT SIGNIFICANT (After FDR correction)")
        continue

    # Post Hoc Test: Games-Howell
    # Note: Games-Howell already adjusts for multiple pairwise comparisons
    # WITHIN this specific context, so we use its output as-is.
    df_ph = pg.pairwise_gameshowell(data=res["context"], dv=dv, between=between)

    for _, row in df_ph.iterrows():  # noqa: F402
        if row["B"] != "oracle":
            continue
        if row["pval"] > significance_level:
            continue

        tuple_ = (str(res["keys"][0]), str(res["keys"][1]), str(row["A"]))
        if row["T"] >= 0:
            significantly_better.add(tuple_)
        else:
            significantly_worse.add(tuple_)

print("Significantly Better than Oracle:")
for item in significantly_better:
    print(item)

print("\nSignificantly Worse than Oracle:")
for item in significantly_worse:
    print(item)

0 ('abrupt', 'EWC') NOT SIGNIFICANT (After FDR correction)
4 ('abrupt', 'RWalk') NOT SIGNIFICANT (After FDR correction)
5 ('abrupt', 'SI') NOT SIGNIFICANT (After FDR correction)
11 ('gradual', 'SI') NOT SIGNIFICANT (After FDR correction)
16 ('slow', 'RWalk') NOT SIGNIFICANT (After FDR correction)
17 ('slow', 'SI') NOT SIGNIFICANT (After FDR correction)
Significantly Better than Oracle:
('gradual', 'LWF', 'ADWIN')
('gradual', 'LWF', 'ADWIN*')

Significantly Worse than Oracle:
('gradual', 'RWalk', 'ADWIN')
('gradual', 'EWC', 'ADWIN*')
('slow', 'LWF', 'ADWIN')
('slow', 'MAS', 'ADWIN*')


In [108]:
# Generate Table
from pathlib import Path

BOUNDARY_AGNOSTIC = {"FT"}

AGGREGATIONS = ("mean", "std")

df_grouped = df_ev.groupby(
    ["boundary", "strategy", "detector", "detector_label"]
).aggregate(AGGREGATIONS)


def latex_table(df: pd.DataFrame, boundary) -> pd.DataFrame:
    df = df.copy()
    # index to columns
    df.reset_index(inplace=True)
    # Move FT rows to the end of the table, since it is boundary-agnostic
    boundary_agnostic_mask = df["strategy"].isin(BOUNDARY_AGNOSTIC)
    df = pd.concat(
        [df[~boundary_agnostic_mask], df[boundary_agnostic_mask]], ignore_index=True
    )
    # Keep original detector labels for significance lookup.
    df["_detector_label_raw"] = df["detector_label"]

    def best_row_label(row: pd.Series) -> str:
        (detector_label,) = row["detector_label"].values
        (detector,) = row["detector"].values

        if detector_label != "Best":
            return row["detector_label"]

        return f"Best ({detector})"

    df["detector_label"] = df.apply(best_row_label, axis=1)

    # Ignore drift detection metrics (dd.*) for the oracle detector, since they are not
    # meaningful
    oracle_mask = df["detector"] == "oracle"
    dd_cols = [col for col in df.columns if col[0].startswith("dd.")]
    df.loc[oracle_mask, dd_cols] = np.nan

    ndigits_percent = 1
    UP = r"$\uparrow$"
    DOWN = r"$\downarrow$"

    def significance_arrow(row: pd.Series) -> str:
        (strategy,) = row["strategy"].values
        (_detector_label_raw,) = row["_detector_label_raw"].values

        key = (
            str(boundary),
            str(strategy),
            str(_detector_label_raw),
        )
        print(key)
        if key in significantly_better:
            return UP
        if key in significantly_worse:
            return DOWN
        return r"$\ \;$"

    def detector_label_with_significance(row: pd.Series) -> str:
        (label,) = row["detector_label"].values
        label = label.replace("ADWIN_JOINT", "ADWIN*").replace("oracle", "Oracle")
        arrow = significance_arrow(row)
        return f"{arrow}{label}"

    # Create a copy of the data frame with just the index
    df_formatted = pd.DataFrame(index=df.index)
    df_formatted["Strategy"] = df["strategy"]
    df_formatted["Detector"] = df.apply(detector_label_with_significance, axis=1)

    df_formatted[f"Acc. {UP}"] = format_column(
        df["ocl.accuracy_seen_avg"] * 100,
        max_style="textbf",
        ndigits=ndigits_percent,
    )
    df_formatted[r"Acc$_{ttt}$ " + UP] = format_column(
        df["ttt.accuracy"],
        max_style="textbf",
        ndigits=ndigits_percent,
    )
    df_formatted[f"FWT {UP}"] = format_column(
        df["ocl.forward_transfer"] * 100,
        max_style="textbf",
        ndigits=ndigits_percent,
    )
    df_formatted[f"BWT {UP}"] = format_column(
        df["ocl.backward_transfer"] * 100,
        max_style="textbf",
        ndigits=ndigits_percent,
    )
    df_formatted[f"F1 {UP}"] = format_column(
        df[F1_METRIC] * 100,
        max_style="textbf",
        ndigits=ndigits_percent,
    )
    df_formatted[f"NMDT {DOWN}"] = format_column(
        df["nmdt"],
        min_style="textbf",
        ndigits=2,
    )

    df_formatted.set_index(["Strategy", "Detector"], inplace=True)
    return df_formatted


for b in sorted(df_ev["boundary"].unique()):
    _table_df = latex_table(df_grouped.loc[b], b)
    _table_latex = _table_df.to_latex(
        # index=False,
        escape=False,
        sparsify=True,
        na_rep=" ",
    )
    # Remove cline from the bottom of the table
    _table_latex = _table_latex.replace(r"\cline{1-8}", "\\midrule")
    _table_latex = _table_latex.replace("\\midrule\n\\bottomrule", "\\bottomrule")
    _table_latex = _table_latex.replace("N/A", "")

    table_path = Path(f"../table/strategy_{b}.tex")
    print(f"Saving {table_path}")
    table_path.write_text(_table_latex)

('abrupt', 'EWC', 'ADWIN')
('abrupt', 'EWC', 'ADWIN*')
('abrupt', 'EWC', 'oracle')
('abrupt', 'LWF', 'ADWIN')
('abrupt', 'LWF', 'ADWIN*')
('abrupt', 'LWF', 'oracle')
('abrupt', 'MAS', 'ADWIN')
('abrupt', 'MAS', 'ADWIN*')
('abrupt', 'MAS', 'oracle')
('abrupt', 'RWalk', 'ADWIN')
('abrupt', 'RWalk', 'ADWIN*')
('abrupt', 'RWalk', 'oracle')
('abrupt', 'SI', 'ADWIN')
('abrupt', 'SI', 'ADWIN*')
('abrupt', 'SI', 'oracle')
('abrupt', 'FT', 'oracle')
Saving ../table/strategy_abrupt.tex
('gradual', 'EWC', 'ADWIN')
('gradual', 'EWC', 'ADWIN*')
('gradual', 'EWC', 'oracle')
('gradual', 'LWF', 'ADWIN')
('gradual', 'LWF', 'ADWIN*')
('gradual', 'LWF', 'oracle')
('gradual', 'MAS', 'ADWIN')
('gradual', 'MAS', 'ADWIN*')
('gradual', 'MAS', 'oracle')
('gradual', 'RWalk', 'ADWIN')
('gradual', 'RWalk', 'ADWIN*')
('gradual', 'RWalk', 'oracle')
('gradual', 'SI', 'ADWIN')
('gradual', 'SI', 'ADWIN*')
('gradual', 'SI', 'oracle')
('gradual', 'FT', 'oracle')
Saving ../table/strategy_gradual.tex
('slow', 'EWC', 'ADWI